# LELA60341: Research Methods in Computational and Corpus Linguistics Week 9

## Huffman coding

Computers store everything, including text, as **bits**. A bit is a single 0 or 1. **Text compression** means storing the same text in fewer bits. This workbook builds up a famous compression method, the **Huffman code**, whose key idea is linguistic at heart: give **frequent** characters **short** codes and **rare** characters **long** ones. It is named after David Huffman, who invented it in 1952 while a student. The workbook uses the Python from the earlier LELA60341 workbooks, including classes (Week 7) and prefixes (Week 8).

Read each explanation, run the example cells, then try the 🏋️ exercises. In an exercise, replace each `???` with your own code, then run the ✅ check cell underneath. When you see a 🔮 **Predict** prompt, decide what you think will happen *before* you run the cell.

In [ ]:
# ▶ Run this cell first. It defines the check() helper used by every exercise.
def check(label, condition):
    """Print a tick or a cross instead of crashing, so you can keep working."""
    try:
        passed = bool(condition() if callable(condition) else condition)
    except Exception as err:
        passed = False
        label += f"  (error: {type(err).__name__}: {err})"
    print(("✅ " if passed else "❌ ") + label)
    return passed

---
# Part 1 · Fixed-length and variable-length codes

Each character is stored as a number (`ord("a")` is 97, Week 8), and the computer stores that number in **binary**. Binary writes numbers using only the digits 0 and 1. Each position is worth twice the one to its right (1, 2, 4, 8, 16, 32, 64, 128), so `01100001` means 64 + 32 + 1 = 97.

**ASCII** (the American Standard Code for Information Interchange) is an early standard that gives each English letter, digit and punctuation mark a number, traditionally stored in **8 bits** (one **byte**) per character. `format(n, "08b")` shows a number in binary, padded to 8 digits. (Throughout, we write bits as strings of `"0"` and `"1"` so that we can see them.)

In [ ]:
for c in "cat":
    print(c, ord(c), format(ord(c), "08b"))

This is a **fixed-length code**: every character's code word has the same length. 8 bits allow 2⁸ = 256 different code words, but most texts use far fewer characters. The sentence `"a fast runner need never be afraid of the dark"` uses only 16 distinct characters, and 4 bits give 2⁴ = 16 code words. So its 46 characters need 46 × 8 = 368 bits in ASCII, but only 46 × 4 = 184 bits with a 4-bit code.

A **variable-length code** can do better still, by giving frequent characters **short** code words and rare characters **long** ones, as Morse code does. Morse code sends messages along telegraph wires as short and long signals (dots and dashes). The most common English letter, `E`, is a single dot, while the rare `Q` is dash–dash–dot–dash. But there is a catch. Suppose `a` is `0`, `b` is `1` and `c` is `01`. Does `01` mean `c`, or `a` followed by `b`? The code is ambiguous, because the code word for `a` is a **prefix** of the code word for `c`.

The solution is a **prefix code**: a code in which **no code word is a prefix of another**. Then, reading the bits from left to right, as soon as the bits read so far form a code word, that character is finished, because no longer code word starts the same way.

---
# Part 2 · Encoding and decoding

Here is a prefix code for the letters of `"abracadabra"`. `a`, the most frequent letter, gets the shortest code word:

In [ ]:
abra_codes = {"a": "0", "b": "110", "r": "111", "c": "100", "d": "101"}

**Encoding** replaces each character with its code word and joins the code words together. **Decoding** reads the bits one at a time, collecting them in a string `buffer`. As soon as `buffer` is a code word, we record its character and empty the buffer. To look up "which character has this code word?", we first build the reverse dict, mapping code words to characters.

### 🏋️ Exercise 1 · Practice — Encode and decode

1. Complete `encode(text, codes)`, which returns the code words of all the characters of `text` joined into one string. Use `"".join` and a list comprehension.
2. Complete `decode(bits, codes)`. The reverse dict and the loop are given.

In [ ]:
def encode(text, codes):
    return ???

def decode(bits, codes):
    char_for = {}                         # the reverse dict: code word -> character
    for c in codes:
        char_for[codes[c]] = c
    result = []
    buffer = ""
    for bit in bits:
        buffer += bit
        if ???:              # the buffer is a complete code word
            result.append(???)    # the character for this code word
            buffer = ""
    return "".join(result)

bits = encode("abracadabra", abra_codes)
print(bits, len(bits), "bits, against", 8 * len("abracadabra"), "in ASCII")
print(decode(bits, abra_codes))

In [ ]:
# ✅ Check
check("encode", lambda: encode("abra", abra_codes) == "01101110")
check("decode", lambda: decode("01101110", abra_codes) == "abra")
check("decoding undoes encoding", lambda: decode(encode("cadabra", abra_codes), abra_codes) == "cadabra")

---
# Part 3 · Huffman's algorithm

Where did `abra_codes` come from? **Huffman's algorithm** builds the best possible prefix code for a text: the one that encodes it in the **fewest bits**. It starts from the **frequency** of each character in the text, and works with **groups** of characters. Each group has a total frequency and a partly built code word for each of its characters.

1. **Start** with one group per character, with that character's frequency and an empty code word `""`.
2. **Repeat** until only one group is left: take out the two groups with the **smallest** frequencies; put a `0` at the front of every code word in the first and a `1` at the front of every code word in the second; and put them back as **one** group, whose frequency is the sum of the two.
3. The last group holds the finished code words.

Rare characters are merged early and often, so they get long code words. Frequent characters are merged late, so theirs stay short. And because two merged groups always get different first bits, no code word can be a prefix of another. For `"abracadabra"` (a 5, b 2, r 2, c 1, d 1):

| step | take out | merged group | frequency |
|---|---|---|---|
| 1 | c (1), d (1) | c `0`, d `1` | 2 |
| 2 | b (2), r (2) | b `0`, r `1` | 4 |
| 3 | {c, d} (2), {b, r} (4) | c `00`, d `01`, b `10`, r `11` | 6 |
| 4 | a (5), {c, d, b, r} (6) | a `0`, c `100`, d `101`, b `110`, r `111` | 11 |

In Python, we store each group as a tuple `(frequency, codes)`, where `codes` is a dict mapping each of its characters to its code word so far. The helper `pop_smallest(groups)` removes and returns the group with the smallest frequency. (`groups.pop(i)` removes and returns the element at position `i`.)

In [ ]:
def frequencies(text):
    freq = {}
    for c in text:
        freq[c] = freq.get(c, 0) + 1          # the counting pattern from Week 3
    return freq

def pop_smallest(groups):
    smallest = 0                                # the position of the smallest group so far
    for i in range(len(groups)):
        if groups[i][0] < groups[smallest][0]:  # compare the frequencies
            smallest = i
    return groups.pop(smallest)

print(frequencies("abracadabra"))

### 🏋️ Exercise 2 · Practice — Huffman's algorithm

Complete the merge in `huffman_codes(text)`:

1. add `"0"` to the front of each code word in `codes1` and `"1"` to the front of each code word in `codes2`, putting the results in the new dict `merged`
2. add the merged group, a tuple of its total frequency and `merged`, back into `groups`

The `print` line shows each step, so you can compare it with the table above.

In [ ]:
def huffman_codes(text, show_steps=False):
    freq = frequencies(text)
    groups = []
    for c in freq:
        groups.append((freq[c], {c: ""}))            # 1. one group per character
    while len(groups) > 1:                           # 2. until one group is left
        f1, codes1 = pop_smallest(groups)
        f2, codes2 = pop_smallest(groups)
        merged = {}
        for c in codes1:
            merged[c] = ???
        for c in codes2:
            merged[c] = ???
        groups.append(???)
        if show_steps:
            print(f1, "+", f2, "->", merged)
    return groups[0][1]                              # 3. the codes of the last group

print(huffman_codes("abracadabra", show_steps=True))

In [ ]:
# ✅ Check
check("abracadabra gives abra_codes", lambda: huffman_codes("abracadabra") == abra_codes)
check("every character gets a code word", lambda: set(huffman_codes("hello world")) == set("hello world"))

🔮 Now a longer example: the sentence from Part 1. Which character will get the shortest code word?

In [ ]:
X = "a fast runner need never be afraid of the dark"
freq = frequencies(X)
codes = huffman_codes(X)

pairs = sorted([(len(codes[c]), c) for c in codes])    # (code length, character), shortest first
for length, c in pairs:
    print(repr(c), freq[c], codes[c])
print("ASCII:", 8 * len(X), "bits   4-bit code:", 4 * len(X), "bits   Huffman:", len(encode(X, codes)), "bits")

The space, the most frequent character, gets a 2-bit code word, and characters that occur once get 5 or 6 bits. Huffman's code needs only 165 bits. Other versions of this example sometimes show different code words for some characters: when two groups have the **same** frequency, either may be taken out first, and different choices give different code words, but always the same total number of bits.

---
# Part 4 · The greedy method

Huffman's algorithm is an example of the **greedy method**: build an answer through a series of choices, at each step making the choice that looks best **right now** (here, merging the two rarest groups). This doesn't always give the best overall answer. Paying 6p with as few coins as possible from 1p, 3p and 4p coins, a greedy cashier takes 4p + 1p + 1p, three coins, but 3p + 3p uses only two. Problems where the greedy method *does* give the best answer have the **greedy-choice property**, and finding the shortest prefix code for a text is one of them, so Huffman's code is always optimal.

---
# Part 5 · A `HuffmanCode` class

Let's package everything into a class (Week 7). A `HuffmanCode` object is built from a sample text and keeps the code in an instance variable. Note that whoever decodes the bits needs the **same** code, so a real compressed file stores the code alongside the bits.

### 🏋️ Exercise 3 · Practice — Complete the class

The constructor is done for you. Complete `encode(self, text)` and `decode(self, bits)`, which use the functions `encode` and `decode` from Part 2 with `self._codes`.

In [ ]:
class HuffmanCode:
    """A Huffman code built from a sample text."""

    def __init__(self, sample):
        self._codes = huffman_codes(sample)

    def encode(self, text):
        return ???

    def decode(self, bits):
        return ???

firth = ("you shall know a word by the company it keeps. "
         "that is the idea at the heart of distributional semantics.")
hc = HuffmanCode(firth)
bits = hc.encode(firth)
print(bits[:60], "...")
print(hc.decode(bits))
print(len(bits), "bits, against", 8 * len(firth), "in ASCII")

In [ ]:
# ✅ Check
check("decode undoes encode", lambda: hc.decode(hc.encode(firth)) == firth)
check("encode uses the code", lambda: HuffmanCode("abracadabra").encode("abra") == "01101110")

---
## ✏️ Summary

You can now explain why fixed-length codes waste space; explain what a **prefix code** is and why it makes decoding unambiguous; encode and decode text with a prefix code; carry out **Huffman's algorithm** by repeatedly merging the two rarest groups; and describe the **greedy method**.

*These materials are adapted from M. T. Goodrich, R. Tamassia and M. H. Goldwasser, *Data Structures and Algorithms in Python* (Wiley, 2013).*